# Heat Equation with Gaussian Initial Condition

This notebook demonstrates the 1D heat equation:

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

For an explicit scheme in 1D, stability is controlled by:

$$\frac{\alpha\,dt}{dx^2} \le 0.5$$

where:
- $\alpha$ is thermal diffusivity
- $dt$ is the time step
- $dx$ is the spatial spacing

This notebook shows:
1. a stable simulation that runs normally,
2. an unstable parameter choice that is caught by validation (without running an exploding simulation).

In [ ]:
# Setup imports
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

workspace_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pde_framework").exists():
        workspace_root = candidate
        break

if workspace_root is None:
    raise RuntimeError("Could not locate workspace root containing pde_framework")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import pde_framework as pde

importlib.reload(pde)

from pde_framework import (
    DirichletBC,
    ExplicitEulerIntegrator,
    Grid1D,
    HeatEquation,
    ScalarField,
    Solver,
    gaussian_1d,
    validate_heat_cfl_1d,
 )

print("Imported pde_framework from", workspace_root)

## 1. Domain and Initial Condition

Create a 1D domain and initialize a Gaussian temperature profile.

In [ ]:
# Domain and initial condition
grid = Grid1D(-1.0, 1.0, 201)
u0 = ScalarField(grid)
u0.apply_function(lambda x: gaussian_1d(x, center=0.0, sigma=0.2, amplitude=1.0))

print(f"grid points: {grid.n_points}, dx={grid.dx:.5f}")
print(f"initial max(u): {u0.data.max():.5f}")

## 2. Boundary Conditions

Use homogeneous Dirichlet boundary conditions: $u(x_{min})=u(x_{max})=0$.

In [ ]:
# Boundary condition object
bc = DirichletBC(left_value=0.0, right_value=0.0)
print(bc)

## 3. Stable Configuration and Solver

Choose a stable $dt$ so that $\alpha dt / dx^2 \le 0.5$.

In [ ]:
# Stable simulation setup
alpha = 0.1
dt_stable = 0.4 * grid.dx**2 / alpha  # 80% of the 0.5 CFL threshold
n_steps = 500
save_every = 100

stable_cfl = alpha * dt_stable / (grid.dx**2)
print(f"stable CFL ratio alpha*dt/dx^2 = {stable_cfl:.4f}")
assert validate_heat_cfl_1d(alpha=alpha, dt=dt_stable, dx=grid.dx)

equation = HeatEquation(alpha=alpha, bc=bc)
integrator = ExplicitEulerIntegrator()
solver = Solver(integrator, equation)

## 4. Run Stable Simulation

In [ ]:
# Run stable simulation
u0_stable = u0.copy()
bc.apply(u0_stable.data, u0_stable.grid)

stable_snapshots = solver.run(
    u0_stable,
    dt=dt_stable,
    n_steps=n_steps,
    save_every=save_every,
 )

print(f"stored snapshots: {len(stable_snapshots)}")
print(f"final max(u): {stable_snapshots[-1].data.max():.5f}")

## 5. Visualize Stable Evolution

In [ ]:
# Plot selected stable snapshots
fig, ax = plt.subplots(figsize=(10, 5))

for idx, field in enumerate(stable_snapshots):
    step = idx * save_every
    ax.plot(grid.x, field.data, label=f"step {step}")

ax.set_title("Stable heat-equation evolution (Explicit Euler)")
ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.grid(alpha=0.3)
ax.legend(ncol=2)
plt.show()

## 6. Validation and Unstable `dt` Demonstration

We verify expected behavior for the stable run and then demonstrate that an unstable time-step is rejected by validation before any unstable integration is executed.

In [ ]:
# Stable run checks
max_values = [s.data.max() for s in stable_snapshots]
assert all(
    max_values[i] >= max_values[i + 1] - 1e-12
    for i in range(len(max_values) - 1)
), "Expected diffusive decay of maximum temperature"

for s in stable_snapshots:
    assert np.isclose(s.data[0], 0.0), "Left Dirichlet BC violated"
    assert np.isclose(s.data[-1], 0.0), "Right Dirichlet BC violated"

# Unstable dt demonstration caught by validator (no exploding run)
dt_unstable = 0.6 * grid.dx**2 / alpha
ratio_unstable = alpha * dt_unstable / (grid.dx**2)
print(f"unstable CFL ratio alpha*dt/dx^2 = {ratio_unstable:.4f}")

try:
    validate_heat_cfl_1d(alpha=alpha, dt=dt_unstable, dx=grid.dx)
except ValueError as exc:
    print("Validation correctly rejected unstable dt:")
    print(exc)
else:
    raise AssertionError("Expected unstable dt to be rejected by validate_heat_cfl_1d")

try:
    solver.run(u0, dt=dt_unstable, n_steps=1)
except ValueError as exc:
    print("Solver hook also rejected unstable dt:")
    print(exc)
else:
    raise AssertionError("Expected solver to reject unstable dt via equation hook")

print("All Epic 6 notebook checks passed.")

## 7. Boundary-Condition Comparison (Epic 7)

We compare heat diffusion from the same Gaussian initial condition under multiple boundary conditions.

- **Dirichlet** fixes boundary values.
- **Neumann** fixes boundary gradients (here zero-gradient).
- **Periodic** wraps the domain so left and right edges are connected.
- **Robin** mixes value and gradient at the boundaries.

Below, assertions verify that each BC applies expected boundary behavior before time integration.

In [ ]:
# Compare multiple boundary conditions (Dirichlet, Neumann, Periodic, Robin)
import importlib
import pde_framework.calculation_core as calculation_core

importlib.reload(calculation_core)

from pde_framework.boundary_conditions.clients import NeumannBC, PeriodicBC, RobinBC

bc_list = [
    ("Dirichlet", DirichletBC(0.0, 0.0)),
    ("Neumann (zero grad)", NeumannBC(0.0, 0.0)),
    ("Periodic", PeriodicBC()),
    (
        "Robin (mixed)",
        RobinBC(left_a=1.0, left_b=0.5, left_c=0.0, right_a=1.0, right_b=0.5, right_c=0.0),
    ),
]

final_fields = {}
boundary_checks = {}

for name, bc_obj in bc_list:
    u_init = u0.copy()
    bc_obj.apply(u_init.data, u_init.grid)

    if name == "Dirichlet":
        assert np.isclose(u_init.data[0], 0.0)
        assert np.isclose(u_init.data[-1], 0.0)
        boundary_checks[name] = "u[0]=0 and u[-1]=0"
    elif name.startswith("Neumann"):
        assert np.isclose(u_init.data[0], u_init.data[1])
        assert np.isclose(u_init.data[-1], u_init.data[-2])
        boundary_checks[name] = "u[0]=u[1] and u[-1]=u[-2]"
    elif name == "Periodic":
        assert np.isclose(u_init.data[0], u_init.data[-2])
        assert np.isclose(u_init.data[-1], u_init.data[1])
        boundary_checks[name] = "u[0]=u[-2] and u[-1]=u[1]"
    else:
        # Robin mixed BC check via boundary residuals a*u + b*du/dn - c ≈ 0
        left_res = 1.0 * u_init.data[0] + 0.5 * (u_init.data[1] - u_init.data[0]) / grid.dx - 0.0
        right_res = 1.0 * u_init.data[-1] + 0.5 * (u_init.data[-1] - u_init.data[-2]) / grid.dx - 0.0
        assert np.isclose(left_res, 0.0, atol=1e-10)
        assert np.isclose(right_res, 0.0, atol=1e-10)
        boundary_checks[name] = "a*u+b*du/dn=c residuals near zero"

    eq = HeatEquation(alpha=alpha, bc=bc_obj)
    solver_bc = Solver(integrator, eq)
    snaps = solver_bc.run(u_init, dt=dt_stable, n_steps=200, save_every=200)
    final_fields[name] = snaps[-1]

print("Collected final states:", list(final_fields.keys()))
print("Boundary checks:")
for key, value in boundary_checks.items():
    print(f"- {key}: {value}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, field in final_fields.items():
    ax.plot(grid.x, field.data, label=name)
ax.set_title("Boundary condition comparison (final state)")
ax.set_xlabel("x")
ax.set_ylabel("u(x)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 8. Time Integrator Comparison: Explicit Euler vs. RK4

We now compare the temporal accuracy of explicit Euler with the 4-stage Runge-Kutta integrator (RK4).

- **Explicit Euler**: 1st-order in time; requires smaller `dt` for accuracy.
- **RK4**: 4th-order in time; provides higher accuracy with the same (or larger) time step.

Below, we run the same heat diffusion problem with both integrators and compare the solutions.

In [ ]:
# Import RK4Integrator
from pde_framework import RK4Integrator

# Set up Euler and RK4 integrators with same problem
bc_dirichlet = DirichletBC(left_value=0.0, right_value=0.0)
eq_euler = HeatEquation(alpha=alpha, bc=bc_dirichlet)
eq_rk4 = HeatEquation(alpha=alpha, bc=bc_dirichlet)

integrator_euler = ExplicitEulerIntegrator()
integrator_rk4 = RK4Integrator()

solver_euler = Solver(integrator_euler, eq_euler)
solver_rk4 = Solver(integrator_rk4, eq_rk4)

# Initial condition (same for both)
u_euler = u0.copy()
u_rk4 = u0.copy()

# Time integration parameters
n_euler_steps = 500
dt_euler = 0.4 * grid.dx**2 / alpha  # Stable CFL

# RK4 can typically use larger dt while maintaining stability
# For comparison, use same dt as Euler
dt_rk4 = dt_euler
n_rk4_steps = 500

# Run both solvers
print("Running Euler integrator...")
snapshots_euler = solver_euler.run(
    u_euler, dt=dt_euler, n_steps=n_euler_steps, save_every=100
)

print("Running RK4 integrator...")
snapshots_rk4 = solver_rk4.run(
    u_rk4, dt=dt_rk4, n_steps=n_rk4_steps, save_every=100
)

print(f"Euler snapshots: {len(snapshots_euler)}")
print(f"RK4 snapshots: {len(snapshots_rk4)}")
print(f"Euler final max(u): {snapshots_euler[-1].data.max():.6f}")
print(f"RK4   final max(u): {snapshots_rk4[-1].data.max():.6f}")

In [ ]:
# Visualize Euler and RK4 evolution side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Euler plot
ax = axes[0]
for idx, field in enumerate(snapshots_euler):
    step = idx * 100
    ax.plot(grid.x, field.data, label=f"step {step}", alpha=0.7)
ax.set_title("Euler Integrator Evolution")
ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.grid(alpha=0.3)
ax.legend(ncol=2, fontsize=8)

# RK4 plot
ax = axes[1]
for idx, field in enumerate(snapshots_rk4):
    step = idx * 100
    ax.plot(grid.x, field.data, label=f"step {step}", alpha=0.7)
ax.set_title("RK4 Integrator Evolution")
ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.grid(alpha=0.3)
ax.legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Compare solutions at each snapshot
differences = []
for euler_snap, rk4_snap in zip(snapshots_euler, snapshots_rk4):
    diff = np.linalg.norm(euler_snap.data - rk4_snap.data)
    differences.append(diff)

# Plot difference evolution
fig, ax = plt.subplots(figsize=(10, 5))
steps = np.arange(len(differences)) * 100
ax.semilogy(steps, differences, "o-", linewidth=2, markersize=6)
ax.set_title("L2 Norm Difference: RK4 - Euler")
ax.set_xlabel("Time step")
ax.set_ylabel("||u_RK4 - u_Euler||_2")
ax.grid(alpha=0.3, which="both")
plt.show()

print(f"Max difference: {max(differences):.6e}")
print(f"Final difference: {differences[-1]:.6e}")

# Validate both solutions preserve boundary conditions
for snap_name, snapshots in [("Euler", snapshots_euler[1:]), ("RK4", snapshots_rk4[1:])]:
    for snap in snapshots:
        assert np.isclose(snap.data[0], 0.0, atol=1e-1), f"{snap_name} violated left BC"
        assert np.isclose(snap.data[-1], 0.0, atol=1e-1), f"{snap_name} violated right BC"
print("Both integrators correctly maintained Dirichlet BCs throughout integration.")

# Summary
print("\n=== Epic 9 Euler vs. RK4 Comparison ===")
print(f"Spatial grid: {grid.n_points} points, dx={grid.dx:.5f}")
print(f"Time steps: {n_euler_steps} steps, dt={dt_euler:.5f}")
print(f"Euler  final max: {snapshots_euler[-1].data.max():.6f}")
print(f"RK4    final max: {snapshots_rk4[-1].data.max():.6f}")
print(f"Difference norm: {differences[-1]:.6e}")
print("Spatial discretization is identical; only temporal integration differs.")
print("Epic 9 notebook checks passed.")